# 06 · Simulación de producción

**Pregunta:** ¿cómo se armó la producción simulada y qué respondió la API en cada lote?

- Patrón **batch semanal**: cada lote de simulación es una semana de cocción (sección 5 del PSet 4).
- Lotes 1 a 9: los 45 lotes reales del conjunto de prueba del PSet 3 (17/08 al 12/10/2026), posteriores a la fecha de corte del entrenamiento.
- Lote 10: **deriva inducida** (`src/preparar_simulacion.py`): 6 lotes reales con `carga_planta_14d` + 12 y `generacion_levadura` 4 o 5. La merma real no se tocó.
- Las predicciones las hizo la API desde el flujo `kestra/produccion_lotes.yml` y quedaron en `PRODUCCION.PREDICCIONES` (exportada a `docs/predicciones.csv` con `src/exportar_evidencia.py`).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SIM = RAIZ / "data" / "simulacion"
DOCS = RAIZ / "docs"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Los lotes de la simulación

In [ ]:
sim = pd.read_csv(SIM / "lotes_simulacion.csv", dtype={"numero_documento": str})
resumen = (sim.groupby("lote_simulacion")
              .agg(semana=("semana", "first"), deriva=("deriva", "max"),
                   lotes_cerveza=("numero_documento", "size"), merma_alta=("merma_alta", "sum")))
resumen["pct_merma_alta"] = (resumen["merma_alta"] / resumen["lotes_cerveza"]).round(2)
resumen

Las semanas tienen entre 2 y 6 lotes de cerveza (limitación: las métricas por semana son inestables). El 80 % de los lotes de la simulación tuvo merma alta, frente al 64 % del entrenamiento.

## 2. Qué cambió en el lote con deriva

In [ ]:
variables = ["carga_planta_14d", "generacion_levadura", "levadura_nueva", "sin_generacion"]
comparacion = sim.groupby("deriva")[variables].mean().round(2)
comparacion.index = ["lotes normales (1-9)", "lote con deriva (10)"]
comparacion

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
for eje, v in zip(ejes, ["carga_planta_14d", "generacion_levadura"]):
    for d, color, nombre in [(0, "#BFB3A3", "normales"), (1, "#E8A33D", "deriva")]:
        eje.hist(sim.loc[sim["deriva"] == d, v].dropna(), bins=10, alpha=0.8, color=color, label=nombre)
    eje.set_title(v)
    eje.legend()
plt.tight_layout()
plt.show()

## 3. Lo que respondió la API

In [ ]:
pred = pd.read_csv(DOCS / "predicciones.csv", dtype={"numero_documento": str})
pred = pred[pred["lote_simulacion"].notna()]
# Última predicción de cada lote de cerveza (un lote reenviado deja dos filas)
pred = (pred.sort_values("timestamp_prediccion")
            .drop_duplicates(["lote_simulacion", "numero_documento"], keep="last"))
por_lote = (pred.groupby("lote_simulacion")
                .agg(predicciones=("numero_documento", "size"),
                     con_clarificante=("decision", lambda d: (d == "aplicar_clarificante").sum()),
                     probabilidad_media=("probabilidad", "mean"),
                     version_modelo=("version_modelo", "first"),
                     respaldo=("respaldo", "sum"),
                     latencia_p95_ms=("latencia_ms", lambda x: x.quantile(0.95))))
por_lote.round(3)

In [ ]:
fig, eje = plt.subplots(figsize=(9, 3.5))
colores = ["#E8A33D" if c > 2 else "#BFB3A3" for c in por_lote["con_clarificante"]]
eje.bar(por_lote.index, por_lote["con_clarificante"], color=colores)
eje.axhline(2, color="#1E1A16", linestyle="--", label="presupuesto: 2 por semana")
eje.set_xlabel("lote de simulación (semana)")
eje.set_ylabel("lotes con clarificante")
eje.set_title("El umbral fijo supera el presupuesto en 5 de 9 semanas; con deriva recomienda 0")
eje.legend()
plt.tight_layout()
plt.show()

## 4. Valor de negocio de cada política (sección 9)

In [ ]:
valor = pd.read_csv(SIM / "valor_negocio.csv")
valor

**Lectura:** en las semanas normales, el modelo con umbral fijo baja el costo de los errores de 4.272 a 1.607 USD, pero excede el presupuesto; con el tope de 2 por semana ahorra 1.566 USD, 330 más que la regla por estilo. En la semana con deriva, solo el top-2 ahorra algo (279 USD).